In [ ]:
import os
import glob
from ultralytics import YOLO

def intersection_over_union(boxA, boxB):
    """
    Calculates the Intersection over Union (IoU) of two bounding boxes.
    Boxes are expected in the format [x1, y1, x2, y2].
    """
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    # Compute the area of intersection
    interArea = max(0, xB - xA) * max(0, yB - yA)
    
    # Compute the area of both bounding boxes
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    
    if float(boxAArea + boxBArea - interArea) == 0:
        return 0.0
        
    # Return the IoU ratio
    return interArea / float(boxAArea + boxBArea - interArea)

def convert_yolo_to_coords(x_center, y_center, width, height, img_w=416, img_h=416):
    """
    Converts YOLO format (normalized center x, center y, width, height) 
    to standard bounding box coordinates [x1, y1, x2, y2] based on image size.
    """
    x1 = (x_center - width / 2) * img_w
    y1 = (y_center - height / 2) * img_h
    x2 = (x_center + width / 2) * img_w
    y2 = (y_center + height / 2) * img_h
    return [x1, y1, x2, y2]

def evaluate_model_manually(model_path, images_dir, labels_dir, target_class_in_model, target_class_in_dataset=0, conf_thres=0.45, iou_thres=0.5):
    """
    Evaluates a YOLO model manually against ground truth labels, mapping specific classes.
    Calculates Precision and Recall for a specific target class.
    """
    print(f"Loading model: {model_path}...")
    model = YOLO(model_path)
    
    # Get all jpg and png images in the directory
    image_paths = glob.glob(os.path.join(images_dir, "*.jpg")) + glob.glob(os.path.join(images_dir, "*.png"))
    
    true_positives = 0
    false_positives = 0
    false_negatives = 0

    for img_path in image_paths:
        img_name = os.path.basename(img_path)
        label_path = os.path.join(labels_dir, os.path.splitext(img_name)[0] + ".txt")
        
        # Run inference
        results = model.predict(img_path, conf=conf_thres, verbose=False)
        
        # Fetch original image dimensions from the prediction result
        img_h, img_w = results[0].orig_shape
        
        # Parse Ground Truth (GT) boxes from the label file
        gt_boxes = []
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    class_id = int(parts[0])
                    
                    # Filter only the target class defined in the dataset
                    if class_id == target_class_in_dataset:
                        gt_boxes.append(convert_yolo_to_coords(
                            float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4]), img_w, img_h
                        ))
        
        # Parse predicted boxes from the model results
        pred_boxes = []
        for box in results[0].boxes:
            # Filter only the target class as recognized by the model
            if int(box.cls[0]) == target_class_in_model:
                pred_boxes.append(box.xyxy[0].cpu().numpy())
                
        # Compare predicted boxes with Ground Truth boxes using IoU
        matched_gt = set()
        for p_box in pred_boxes:
            best_iou = 0
            best_gt_idx = -1
            
            # Find the best matching ground truth box
            for idx, g_box in enumerate(gt_boxes):
                if idx in matched_gt:
                    continue  # Skip already matched ground truth boxes
                iou = intersection_over_union(p_box, g_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = idx
            
            # Classify detection as True Positive or False Positive
            if best_iou >= iou_thres:
                true_positives += 1
                matched_gt.add(best_gt_idx)
            else:
                false_positives += 1
                
        # Calculate False Negatives (unmatched ground truth boxes)
        false_negatives += (len(gt_boxes) - len(matched_gt))

    # Compute final metrics
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    
    # Print results
    print("-" * 40)
    print("Evaluation Results:")
    print(f"Precision: {precision:.4f}  (True Positives: {true_positives}, False Positives: {false_positives})")
    print(f"Recall:    {recall:.4f}  (False Negatives [Missed]: {false_negatives})")
    print("-" * 40)
    
    return precision, recall

if __name__ == "__main__":
    # Define paths to validation dataset
    val_images = r"Traffic-1\valid\images"
    val_labels = r"Traffic-1\valid\labels"
    
    # Base YOLOv8 model detects 'car' as class 2, but in the dataset 'car' is class 1
    print("\nEvaluating Base YOLOv8 Model:")
    evaluate_model_manually("yolov8n.pt", val_images, val_labels, target_class_in_model=2, target_class_in_dataset=1)
    
    # Custom V3 model detects 'car' as class 0, and in the dataset 'car' is class 1
    print("\nEvaluating Custom V3 Model:")
    evaluate_model_manually(r"runs\detect\Modele_Carla\osobowki_model_v3\weights\best.pt", val_images, val_labels, target_class_in_model=0, target_class_in_dataset=1)